In [60]:
import requests
import pandas as pd
import time
from datetime import datetime

In [61]:
BASE = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball"
SCHEDULE_URL = f"{BASE}/teams/{{team_id}}/schedule"
BOXSCORE_URL = f"{BASE}/summary?event={{game_id}}"

In [62]:
SEASON_YEARS_TO_TRY = [2026, 2025]

In [80]:
TOURNAMENT_FIELD = {
    # EAST REGION
    "Duke":              {"seed": 1,  "region": "East",    "espn_id": 150},
    "UConn":             {"seed": 2,  "region": "East",    "espn_id": 41},
    "Michigan State":    {"seed": 3,  "region": "East",    "espn_id": 127},
    "Kansas":            {"seed": 4,  "region": "East",    "espn_id": 2305},
    "St. John's":        {"seed": 5,  "region": "East",    "espn_id": 2599},
    "Louisville":        {"seed": 6,  "region": "East",    "espn_id": 97},
    "UCLA":              {"seed": 7,  "region": "East",    "espn_id": 26},
    "Ohio State":        {"seed": 8,  "region": "East",    "espn_id": 194},
    "TCU":               {"seed": 9,  "region": "East",    "espn_id": 2628},
    "UCF":               {"seed": 10, "region": "East",    "espn_id": 2116},
    "South Florida":     {"seed": 11, "region": "East",    "espn_id": 58},
    "Northern Iowa":     {"seed": 12, "region": "East",    "espn_id": 2460},
    "Cal Baptist":       {"seed": 13, "region": "East",    "espn_id": 2856},
    "North Dakota State":{"seed": 14, "region": "East",    "espn_id": 2449},
    "Furman":            {"seed": 15, "region": "East",    "espn_id": 231},
    "Siena":             {"seed": 16, "region": "East",    "espn_id": 2561},
 
    # WEST REGION
    "Arizona":           {"seed": 1,  "region": "West",    "espn_id": 12},
    "Purdue":            {"seed": 2,  "region": "West",    "espn_id": 2509},
    "Gonzaga":           {"seed": 3,  "region": "West",    "espn_id": 2250},
    "Arkansas":          {"seed": 4,  "region": "West",    "espn_id": 8},
    "Wisconsin":         {"seed": 5,  "region": "West",    "espn_id": 275},
    "BYU":               {"seed": 6,  "region": "West",    "espn_id": 252},
    "Miami (FL)":        {"seed": 7,  "region": "West",    "espn_id": 2390},
    "Villanova":         {"seed": 8,  "region": "West",    "espn_id": 222},
    "Utah State":        {"seed": 9,  "region": "West",    "espn_id": 328},
    "Missouri":          {"seed": 10, "region": "West",    "espn_id": 142},
    "Texas":             {"seed": 11, "region": "West",    "espn_id": 251},
    "NC State":          {"seed": 11, "region": "West",    "espn_id": 152},
    "High Point":        {"seed": 12, "region": "West",    "espn_id": 2272},
    "Hawaii":            {"seed": 13, "region": "West",    "espn_id": 62},
    "Kennesaw State":    {"seed": 14, "region": "West",    "espn_id": 338},
    "Queens":            {"seed": 15, "region": "West",    "espn_id": 3101},
    "Long Island University Sharks":               {"seed": 16, "region": "West",    "espn_id": 112358},
 
    # MIDWEST REGION
    "Michigan":          {"seed": 1,  "region": "Midwest", "espn_id": 130},
    "Iowa State":        {"seed": 2,  "region": "Midwest", "espn_id": 66},
    "Virginia":          {"seed": 3,  "region": "Midwest", "espn_id": 258},
    "Alabama":           {"seed": 4,  "region": "Midwest", "espn_id": 333},
    "Texas Tech":        {"seed": 5,  "region": "Midwest", "espn_id": 2641},
    "Tennessee":         {"seed": 6,  "region": "Midwest", "espn_id": 2633},
    "Kentucky":          {"seed": 7,  "region": "Midwest", "espn_id": 96},
    "Georgia":           {"seed": 8,  "region": "Midwest", "espn_id": 61},
    "Saint Louis":       {"seed": 9,  "region": "Midwest", "espn_id": 139},
    "Santa Clara":       {"seed": 10, "region": "Midwest", "espn_id": 2491},
    "SMU":               {"seed": 11, "region": "Midwest", "espn_id": 2567},
    "Miami (OH)":        {"seed": 11, "region": "Midwest", "espn_id": 193},
    "Akron":             {"seed": 12, "region": "Midwest", "espn_id": 2006},
    "Hofstra":           {"seed": 13, "region": "Midwest", "espn_id": 2275},
    "Wright State":      {"seed": 14, "region": "Midwest", "espn_id": 2750},
    "Tennessee State":   {"seed": 15, "region": "Midwest", "espn_id": 2634},
    "UMBC":              {"seed": 16, "region": "Midwest", "espn_id": 2378},
    "Howard":            {"seed": 16, "region": "Midwest", "espn_id": 47},
 
    # SOUTH REGION
    "Florida":           {"seed": 1,  "region": "South",   "espn_id": 57},
    "Houston":           {"seed": 2,  "region": "South",   "espn_id": 248},
    "Illinois":          {"seed": 3,  "region": "South",   "espn_id": 356},
    "Nebraska":          {"seed": 4,  "region": "South",   "espn_id": 158},
    "Vanderbilt":        {"seed": 5,  "region": "South",   "espn_id": 238},
    "North Carolina":    {"seed": 6,  "region": "South",   "espn_id": 153},
    "Saint Mary's":      {"seed": 7,  "region": "South",   "espn_id": 2608},
    "Clemson":           {"seed": 8,  "region": "South",   "espn_id": 228},
    "Iowa":              {"seed": 9,  "region": "South",   "espn_id": 2294},
    "Texas A&M":         {"seed": 10, "region": "South",   "espn_id": 245},
    "VCU":               {"seed": 11, "region": "South",   "espn_id": 2670},
    "McNeese":           {"seed": 12, "region": "South",   "espn_id": 2377},
    "Troy":              {"seed": 13, "region": "South",   "espn_id": 2653},
    "Penn":              {"seed": 14, "region": "South",   "espn_id": 219},
    "Idaho":             {"seed": 15, "region": "South",   "espn_id": 70},
    "Prairie View A&M":  {"seed": 16, "region": "South",   "espn_id": 2504},
    "Lehigh":            {"seed": 16, "region": "South",   "espn_id": 2329},
}

In [64]:
def espn_get(url: str, params: dict = None) -> dict | None:
    """GET request to ESPN API with basic error handling."""
    try:
        r = requests.get(url, params=params, timeout=15)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        print(f"  [API ERROR] {url[:80]}... → {e}")
        return None
 
 
def get_team_game_ids(team_id: int) -> list[str]:
    """
    Fetch a team's schedule and return list of finished game IDs.
    Tries multiple season year values to handle ESPN's convention.
    """
    for year in SEASON_YEARS_TO_TRY:
        url = SCHEDULE_URL.format(team_id=team_id)
        data = espn_get(url, params={"season": year})
        if data is None:
            continue
 
        game_ids = []
        events = data.get("events", [])
        for event in events:
            comps = event.get("competitions", [{}])
            status = comps[0].get("status", {}) if comps else {}
            is_final = status.get("type", {}).get("completed", False)
            if is_final:
                game_ids.append(str(event["id"]))
 
        if game_ids:
            print(f"  → Found {len(game_ids)} completed games (season={year})")
            return game_ids
 
    return []

In [79]:
url = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams"
r = requests.get(url, params={"limit": 500})
data = r.json()

for team in data["sports"][0]["leagues"][0]["teams"]:
    t = team["team"]
    print(f"{t['id']:>5}  {t['displayName']}")

 2000  Abilene Christian Wildcats
 2005  Air Force Falcons
 2006  Akron Zips
 2010  Alabama A&M Bulldogs
  333  Alabama Crimson Tide
 2011  Alabama State Hornets
 2016  Alcorn State Braves
   44  American University Eagles
 2026  App State Mountaineers
    9  Arizona State Sun Devils
   12  Arizona Wildcats
    8  Arkansas Razorbacks
 2032  Arkansas State Red Wolves
 2029  Arkansas-Pine Bluff Golden Lions
  349  Army Black Knights
    2  Auburn Tigers
 2046  Austin Peay Governors
  252  BYU Cougars
 2050  Ball State Cardinals
  239  Baylor Bears
   91  Bellarmine Knights
 2057  Belmont Bruins
 2065  Bethune-Cookman Wildcats
 2066  Binghamton Bearcats
   68  Boise State Broncos
  103  Boston College Eagles
  104  Boston University Terriers
  189  Bowling Green Falcons
   71  Bradley Braves
  225  Brown Bears
 2803  Bryant Bulldogs
 2083  Bucknell Bison
 2084  Buffalo Bulls
 2086  Butler Bulldogs
   13  Cal Poly Mustangs
 2934  Cal State Bakersfield Roadrunners
 2239  Cal State Fullerton

In [65]:
def parse_boxscore(data: dict, game_id: str) -> list[dict]:
    """
    Parse ESPN summary JSON into flat player-game rows.
    Returns a list of dicts, one per player per game.
    """
    rows = []
    boxscore = data.get("boxscore", {})
    players_by_team = boxscore.get("players", [])
 
    # Game metadata
    header = data.get("header", {})
    competitions = header.get("competitions", [{}])
    game_date = competitions[0].get("date", "") if competitions else ""
 
    for team_block in players_by_team:
        team_info = team_block.get("team", {})
        team_name = team_info.get("displayName", "Unknown")
        team_abbr = team_info.get("abbreviation", "")
        team_id = team_info.get("id", "")
 
        # Stats are grouped by category (starters, bench)
        # Column labels are in statistics[0].labels
        stats_sections = team_block.get("statistics", [])
        if not stats_sections:
            continue
 
        labels = stats_sections[0].get("labels", [])
 
        for section in stats_sections:
            athletes = section.get("athletes", [])
            for athlete in athletes:
                player_info = athlete.get("athlete", {})
                stat_values = athlete.get("stats", [])
 
                row = {
                    "game_id": game_id,
                    "game_date": game_date,
                    "team": team_name,
                    "team_abbr": team_abbr,
                    "team_id": team_id,
                    "player_id": player_info.get("id", ""),
                    "player": player_info.get("displayName", ""),
                    "position": player_info.get("position", {}).get("abbreviation", ""),
                    "starter": section.get("type", "") == "starters",
                }
 
                for label, value in zip(labels, stat_values):
                    row[label] = value
 
                rows.append(row)
 
    return rows

In [69]:
def pull_team_boxscores(team_name: str, team_id: int) -> pd.DataFrame | None:
    """
    Pull all game boxscores for a team via ESPN API.
    Returns a DataFrame of player-game-level stats.
    """
    game_ids = get_team_game_ids(team_id)
    if not game_ids:
        return None
 
    all_rows = []
    for i, gid in enumerate(game_ids):
        url = BOXSCORE_URL.format(game_id=gid)
        data = espn_get(url)
        if data:
            rows = parse_boxscore(data, gid)
            all_rows.extend(rows)
 
        # Rate limiting: pause briefly every 10 games
        if (i + 1) % 10 == 0:
            time.sleep(0.5)
 
    if all_rows:
        df = pd.DataFrame(all_rows)
        df["team_query"] = team_name
        return df
 
    return None
 
 
def aggregate_team_stats(box: pd.DataFrame, team_name: str, meta: dict) -> dict:
    """
    Aggregate player-game rows into team-level season averages.
    Filters to only rows matching the queried team (not opponents).
    """
    summary = {
        "team": team_name,
        "seed": meta["seed"],
        "region": meta["region"],
        "espn_id": meta["espn_id"],
    }
 
    # ESPN boxscore includes both teams — filter to just this team
    team_id_str = str(meta["espn_id"])
    team_rows = box[box["team_id"] == team_id_str]
    if team_rows.empty:
        team_rows = box[box["team_query"] == team_name]
    if team_rows.empty:
        return summary
 
    game_ids = team_rows["game_id"].unique()
    summary["games"] = len(game_ids)
 
    # ---- Parsing helpers ----
    def safe_int(val):
        try:
            return int(val)
        except (ValueError, TypeError):
            return 0
 
    def safe_float(val):
        try:
            return float(val)
        except (ValueError, TypeError):
            return 0.0
 
    # Parse each row's stat strings into numeric values
    parsed_rows = []
    for _, row in team_rows.iterrows():
        p = {"game_id": row.get("game_id", "")}
 
        # "made-attempted" columns → split into two fields
        for col, made_key, att_key in [
            ("FG", "FGM", "FGA"),
            ("3PT", "3PM", "3PA"),
            ("FT", "FTM", "FTA"),
        ]:
            if col in row and isinstance(row[col], str) and "-" in row[col]:
                parts = row[col].split("-")
                p[made_key] = safe_int(parts[0])
                p[att_key] = safe_int(parts[1])
 
        # Simple numeric columns
        for col in ["OREB", "DREB", "REB", "AST", "STL", "BLK", "TO", "PF", "PTS"]:
            if col in row:
                p[col] = safe_int(row[col])
 
        # MIN (could be "32" or "32:15")
        if "MIN" in row:
            min_val = str(row["MIN"])
            if ":" in min_val:
                parts = min_val.split(":")
                p["MIN"] = safe_int(parts[0]) + safe_int(parts[1]) / 60
            else:
                p["MIN"] = safe_float(min_val)
 
        parsed_rows.append(p)
 
    if not parsed_rows:
        return summary
 
    parsed_df = pd.DataFrame(parsed_rows)
 
    # Sum per game (player rows → team game totals), then avg across games
    numeric_cols = [c for c in parsed_df.columns if c != "game_id"]
    game_totals = parsed_df.groupby("game_id")[numeric_cols].sum()
 
    for col in numeric_cols:
        summary[f"avg_{col}"] = round(game_totals[col].mean(), 2)
 
    # Shooting percentages
    if summary.get("avg_FGA", 0) > 0:
        summary["FG_PCT"] = round(summary["avg_FGM"] / summary["avg_FGA"], 4)
    if summary.get("avg_3PA", 0) > 0:
        summary["3PT_PCT"] = round(summary["avg_3PM"] / summary["avg_3PA"], 4)
    if summary.get("avg_FTA", 0) > 0:
        summary["FT_PCT"] = round(summary["avg_FTM"] / summary["avg_FTA"], 4)
 
    return summary

In [73]:
teams = list(TOURNAMENT_FIELD.items())
for i, (team_name, meta) in enumerate(teams, 1):
  print(i, team_name, meta["espn_id"])

1 Duke 150
2 UConn 41
3 Michigan State 127
4 Kansas 2305
5 St. John's 2599
6 Louisville 97
7 UCLA 26
8 Ohio State 194
9 TCU 2628
10 UCF 2116
11 South Florida 58
12 Northern Iowa 2460
13 Cal Baptist 2856
14 North Dakota State 2449
15 Furman 231
16 Siena 2561
17 Arizona 12
18 Purdue 2509
19 Gonzaga 2250
20 Arkansas 8
21 Wisconsin 275
22 BYU 252
23 Miami (FL) 2390
24 Villanova 222
25 Utah State 328
26 Missouri 142
27 Texas 251
28 NC State 152
29 High Point 2272
30 Hawaii 62
31 Kennesaw State 338
32 Queens 3101
33 Long Island University 112
34 Michigan 130
35 Iowa State 66
36 Virginia 258
37 Alabama 333
38 Texas Tech 2641
39 Tennessee 2633
40 Kentucky 96
41 Georgia 61
42 Saint Louis 139
43 Santa Clara 2491
44 SMU 2567
45 Miami (OH) 193
46 Akron 2006
47 Hofstra 2275
48 Wright State 2750
49 Tennessee State 2634
50 UMBC 2378
51 Howard 47
52 Florida 57
53 Houston 248
54 Illinois 356
55 Nebraska 158
56 Vanderbilt 238
57 North Carolina 153
58 Saint Mary's 2608
59 Clemson 228
60 Iowa 2294
61 Texa

In [70]:
print("=" * 60)
print("  2026 MARCH MADNESS — ESPN API DATA PULL")
print(f"  Teams: {len(TOURNAMENT_FIELD)}")
print(f"  Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

all_boxscores = []
team_summaries = []
failed_teams = []

teams = list(TOURNAMENT_FIELD.items())

for i, (team_name, meta) in enumerate(teams, 1):
    team_id = meta["espn_id"]
    print(f"\n[{i}/{len(teams)}] {team_name} (#{meta['seed']} {meta['region']}) — ESPN ID {team_id}")

    box = pull_team_boxscores(team_name, team_id)

    if box is not None and not box.empty:
        box["seed"] = meta["seed"]
        box["region"] = meta["region"]
        all_boxscores.append(box)

        summary = aggregate_team_stats(box, team_name, meta)
        team_summaries.append(summary)

        print(f"  ✓ {len(box)} player-game rows | {box['game_id'].nunique()} games")
    else:
        failed_teams.append(team_name)
        print(f"  ✗ No data — verify https://www.espn.com/mens-college-basketball/team/_/id/{team_id}")

    # Rate limiting between teams
    time.sleep(0.3)

  2026 MARCH MADNESS — ESPN API DATA PULL
  Teams: 68
  Started: 2026-03-16 21:42:21

[1/68] Duke (#1 East) — ESPN ID 150
  → Found 34 completed games (season=2026)
  ✓ 965 player-game rows | 34 games

[2/68] UConn (#2 East) — ESPN ID 41
  → Found 34 completed games (season=2026)
  ✓ 1047 player-game rows | 34 games

[3/68] Michigan State (#3 East) — ESPN ID 127
  → Found 32 completed games (season=2026)
  ✓ 974 player-game rows | 32 games

[4/68] Kansas (#4 East) — ESPN ID 2305
  → Found 33 completed games (season=2026)
  ✓ 1030 player-game rows | 33 games

[5/68] St. John's (#5 East) — ESPN ID 2599
  → Found 34 completed games (season=2026)
  ✓ 1041 player-game rows | 34 games

[6/68] Louisville (#6 East) — ESPN ID 97
  → Found 33 completed games (season=2026)
  ✓ 1011 player-game rows | 33 games

[7/68] UCLA (#7 East) — ESPN ID 26
  → Found 34 completed games (season=2026)
  ✓ 1036 player-game rows | 34 games

[8/68] Ohio State (#8 East) — ESPN ID 194
  → Found 33 completed games (s

In [71]:
if all_boxscores:
    combined = pd.concat(all_boxscores, ignore_index=True)

    raw_path = "march_madness_2026_raw_boxscores.csv"
    combined.to_csv(raw_path, index=False)
    print(f"\n✓ Raw boxscores saved: {raw_path}")
    print(f"  {len(combined):,} total rows | {combined['team_query'].nunique()} teams")

if team_summaries:
    summary_df = pd.DataFrame(team_summaries)
    summary_path = "march_madness_2026_team_summary.csv"
    summary_df.to_csv(summary_path, index=False)
    print(f"✓ Team summary saved: {summary_path}")
    print(f"\nSample output:")
    cols_to_show = ["team", "seed", "region", "games"]
    for c in ["avg_PTS", "FG_PCT", "avg_REB", "avg_AST", "avg_TO"]:
        if c in summary_df.columns:
            cols_to_show.append(c)
    print(summary_df[cols_to_show].head(10).to_string(index=False))

if failed_teams:
    print(f"\n⚠ Failed teams ({len(failed_teams)}):")
    for t in failed_teams:
        tid = TOURNAMENT_FIELD[t]["espn_id"]
        print(f"  - {t} → https://www.espn.com/mens-college-basketball/team/_/id/{tid}")

print(f"\n{'=' * 60}")
print(f"  Finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'=' * 60}")


✓ Raw boxscores saved: march_madness_2026_raw_boxscores.csv
  67,560 total rows | 67 teams
✓ Team summary saved: march_madness_2026_team_summary.csv

Sample output:
          team  seed region  games  avg_PTS  FG_PCT  avg_REB  avg_AST  avg_TO
          Duke     1   East     34    82.29  0.4900    37.41    16.94   10.18
         UConn     2   East     34    77.50  0.4824    33.18    18.38   10.32
Michigan State     3   East     32    78.94  0.4706    36.91    18.47   10.81
        Kansas     4   East     33    75.64  0.4533    35.91    14.27   10.27
    St. John's     5   East     34    81.56  0.4539    34.71    16.26   10.06
    Louisville     6   East     33    84.70  0.4703    36.12    17.09   11.33
          UCLA     7   East     34    77.74  0.4708    29.26    16.32    8.29
    Ohio State     8   East     33    79.79  0.4930    29.82    14.36    9.73
           TCU     9   East     33    78.30  0.4479    32.36    15.64   10.36
           UCF    10   East     32    80.97  0.4661   

In [81]:
# --- Pull Long Island University separately and append to CSVs ---
liu_name = "Long Island University Sharks"
liu_meta = TOURNAMENT_FIELD[liu_name]
liu_id = liu_meta["espn_id"]

print(f"Pulling data for {liu_name} (ESPN ID {liu_id})...")

liu_box = pull_team_boxscores(liu_name, liu_id)

if liu_box is not None and not liu_box.empty:
    liu_box["seed"] = liu_meta["seed"]
    liu_box["region"] = liu_meta["region"]
    print(f"  ✓ {len(liu_box)} player-game rows | {liu_box['game_id'].nunique()} games")

    # Append raw boxscores to existing CSV
    liu_box.to_csv(raw_path, mode="a", header=False, index=False)
    print(f"  ✓ Appended raw boxscores to {raw_path}")

    # Build summary and append to existing CSV
    liu_summary = aggregate_team_stats(liu_box, liu_name, liu_meta)
    liu_summary_df = pd.DataFrame([liu_summary])
    liu_summary_df.to_csv(summary_path, mode="a", header=False, index=False)
    print(f"  ✓ Appended team summary to {summary_path}")
else:
    print(f"  ✗ No data for {liu_name} — verify https://www.espn.com/mens-college-basketball/team/_/id/{liu_id}")

Pulling data for Long Island University Sharks (ESPN ID 112358)...
  → Found 34 completed games (season=2026)
  ✓ 1049 player-game rows | 34 games
  ✓ Appended raw boxscores to march_madness_2026_raw_boxscores.csv
  ✓ Appended team summary to march_madness_2026_team_summary.csv
